### Delta log P area under the ablation curve

### Todo
- [ ] Add benchmarks on Qwen


In [2]:
import json
import pandas as pd

# Load results
with open("../results/master_results.json") as f:
    data = json.load(f)
     
# Convert to DataFrame (each experiment = row, metrics = columns)
df = pd.DataFrame(data).T
df.index.name = "experiment"
df = df.reset_index()

# Display as table
df

,experiment,top_k_fraction,avg_delta,variance_delta,sem_delta,accuracy,num_correct,total
0,Llama-3.2-3B__Temperature_lambada_top0.05,0.05,-4.954160,20.151400,0.259174,0.726667,218.0,300.0
1,Llama-3.2-3B__Temperature_lambada_top0.1,0.10,-6.823024,18.421273,0.247799,0.726667,218.0,300.0
2,Llama-3.2-3B__Temperature_lambada_top0.2,0.20,-8.917719,18.778210,0.250188,0.726667,218.0,300.0
3,Llama-3.2-3B__Semantic_lambada_top0.05,0.05,-5.050690,18.781744,0.250212,0.726667,218.0,300.0
4,Llama-3.2-3B__Semantic_lambada_top0.1,0.10,-6.770932,18.357299,0.247368,0.726667,218.0,300.0
...,...,...,...,...,...,...,...,...
64,Qwen2.5-3B__IG_lmbd1000_top0.1,0.10,-8.884273,21.284102,0.145891,0.725000,725.0,1000.0
65,Qwen2.5-3B__IG_lmbd1000_top0.2,0.20,-10.844538,18.922741,0.137560,0.725000,725.0,1000.0
66,Qwen2.5-3B__random_ablation_lmbd1000_top0.05,0.05,-0.750843,4.606579,0.067872,0.725000,725.0,1000.0
67,Qwen2.5-3B__random_ablation_lmbd1000_top0.1,0.10,-1.429132,7.704817,0.087777,0.725000,725.0,1000.0


In [3]:
# Filter lambada experiments and pivot by drop fraction
# lambada_data = {k: v for k, v in data.items() if "lambada" in k.lower() and "3B" in k}
lambada_data = {k: v for k, v in data.items() if "lmbd1000" in k.lower() and "Llama-3.2-1B" in k}

def fmt_val_plus_sem(entry):
    """Format avg_delta ± sem_delta."""
    if entry is None:
        return None
    avg = entry.get("avg_delta")
    sem = entry.get("sem_delta")
    if avg is None:
        return None
    if sem is not None:
        return f"{avg:.2f} ± {sem:.2f}"
    return f"{avg:.2f}"

# Build pivoted table: rows = drop fraction (0.05, 0.1, 0.2), columns = method type
rows = []
for frac in [0.05, 0.1, 0.2]:
    frac_str = f"top{frac}"
    row = {"method": f"{int(frac*100)}%"}
    
    # random drop
    random_key = next((k for k in lambada_data if frac_str in k and "random_ablation" in k), None)
    row["random"] = fmt_val_plus_sem(lambada_data[random_key] if random_key else None)

    grad_key = next((k for k in lambada_data if "IG" in k and frac_str in k), None)
    row["Integrated Grads"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
    
    grad_key = next((k for k in lambada_data if "gradient_x_input" in k and frac_str in k), None)
    row["Input x Grad"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
                        
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Semantic" in k and frac_str in k and "random_drop" not in k), None)
    row["Semantic Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    # top k with temperature scope
    sem_key = next((k for k in lambada_data if "Temperature" in k and frac_str in k and "random_drop" not in k), None)
    row["Temperature Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_4" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)



    rows.append(row)

lambada_table = pd.DataFrame(rows).set_index("method").T
lambada_table

method,5%,10%,20%
random,-0.71 ± 0.06,-1.27 ± 0.08,-2.96 ± 0.11
Integrated Grads,-4.32 ± 0.14,-7.07 ± 0.15,-9.14 ± 0.13
Input x Grad,-6.69 ± 0.15,-8.30 ± 0.15,-9.89 ± 0.13
Semantic Scope,-6.67 ± 0.15,-8.47 ± 0.15,-9.91 ± 0.13
Temperature Scope,-6.79 ± 0.15,-8.66 ± 0.14,-9.94 ± 0.13
Fisher Scope,-6.91 ± 0.15,-8.62 ± 0.14,-10.06 ± 0.13


In [4]:
print(lambada_table.to_markdown())

|                   | 5%           | 10%          | 20%           |
|:------------------|:-------------|:-------------|:--------------|
| random            | -0.71 ± 0.06 | -1.27 ± 0.08 | -2.96 ± 0.11  |
| Integrated Grads  | -4.32 ± 0.14 | -7.07 ± 0.15 | -9.14 ± 0.13  |
| Input x Grad      | -6.69 ± 0.15 | -8.30 ± 0.15 | -9.89 ± 0.13  |
| Semantic Scope    | -6.67 ± 0.15 | -8.47 ± 0.15 | -9.91 ± 0.13  |
| Temperature Scope | -6.79 ± 0.15 | -8.66 ± 0.14 | -9.94 ± 0.13  |
| Fisher Scope      | -6.91 ± 0.15 | -8.62 ± 0.14 | -10.06 ± 0.13 |


In [5]:
# Filter lambada experiments and pivot by drop fraction
lambada_data = {k: v for k, v in data.items() if "lmbd1000" in k.lower() and "Llama-3.2-3B" in k}

def fmt_val_plus_sem(entry):
    """Format avg_delta ± sem_delta."""
    if entry is None:
        return None
    avg = entry.get("avg_delta")
    sem = entry.get("sem_delta")
    if avg is None:
        return None
    if sem is not None:
        return f"{avg:.2f} ± {sem:.2f}"
    return f"{avg:.2f}"

# Build pivoted table: rows = drop fraction (0.05, 0.1, 0.2), columns = method type
rows = []
for frac in [0.05, 0.1, 0.2]:
    frac_str = f"top{frac}"
    row = {"method": f"{int(frac*100)}%"}
    
    # random drop
    random_key = next((k for k in lambada_data if frac_str in k and "random_ablation" in k), None)
    row["random"] = fmt_val_plus_sem(lambada_data[random_key] if random_key else None)

    grad_key = next((k for k in lambada_data if "IG" in k and frac_str in k), None)
    row["Integrated Grads"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
    
    grad_key = next((k for k in lambada_data if "gradient_x_input" in k and frac_str in k), None)
    row["Input x Grad"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
                        
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Semantic" in k and frac_str in k and "random_drop" not in k), None)
    row["Semantic Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    
    # top k with temperature scope
    sem_key = next((k for k in lambada_data if "Temperature" in k and frac_str in k and "random_drop" not in k), None)
    row["Temperature Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_4" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    
    
    rows.append(row)

lambada_table = pd.DataFrame(rows).set_index("method").T
lambada_table

method,5%,10%,20%
random,-0.62 ± 0.06,-1.17 ± 0.08,-2.51 ± 0.10
Integrated Grads,-1.35 ± 0.08,-3.75 ± 0.12,-7.14 ± 0.14
Input x Grad,-4.95 ± 0.14,-7.20 ± 0.14,-9.11 ± 0.13
Semantic Scope,-5.38 ± 0.14,-7.44 ± 0.14,-9.39 ± 0.14
Temperature Scope,-5.40 ± 0.15,-7.48 ± 0.15,-9.53 ± 0.14
Fisher Scope,-5.39 ± 0.14,-7.65 ± 0.14,-9.57 ± 0.13


In [6]:
# Filter lambada experiments and pivot by drop fraction
# lambada_data = {k: v for k, v in data.items() if "lambada" in k.lower() and "3B" in k}
lambada_data = {k: v for k, v in data.items() if "lmbd1000" in k.lower() and "Qwen2.5-3B" in k}

def fmt_val_plus_sem(entry):
    """Format avg_delta ± sem_delta."""
    if entry is None:
        return None
    avg = entry.get("avg_delta")
    sem = entry.get("sem_delta")
    if avg is None:
        return None
    if sem is not None:
        return f"{avg:.2f} ± {sem:.2f}"
    return f"{avg:.2f}"

# Build pivoted table: rows = drop fraction (0.05, 0.1, 0.2), columns = method type
rows = []
for frac in [0.05, 0.1, 0.2]:
    frac_str = f"top{frac}"
    row = {"method": f"{int(frac*100)}%"}
    
    # random drop
    random_key = next((k for k in lambada_data if frac_str in k and "random_ablation" in k), None)
    row["random"] = fmt_val_plus_sem(lambada_data[random_key] if random_key else None)
    
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Semantic" in k and frac_str in k and "random_drop" not in k), None)
    row["Semantic Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    grad_key = next((k for k in lambada_data if "IG" in k and frac_str in k), None)
    row["Integrated Grads"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
    
    grad_key = next((k for k in lambada_data if "gradient_x_input" in k and frac_str in k), None)
    row["Input x Grad"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
                        
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_4" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    
        
    # top k with temperature scope
    sem_key = next((k for k in lambada_data if "Temperature" in k and frac_str in k and "random_drop" not in k), None)
    row["Temperature Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)


    
    rows.append(row)

lambada_table = pd.DataFrame(rows).set_index("method").T
lambada_table

method,5%,10%,20%
random,-0.75 ± 0.07,-1.43 ± 0.09,-3.35 ± 0.13
Semantic Scope,-6.76 ± 0.15,-8.56 ± 0.15,-10.50 ± 0.14
Integrated Grads,-7.11 ± 0.15,-8.88 ± 0.15,-10.84 ± 0.14
Input x Grad,-7.11 ± 0.15,-8.93 ± 0.15,-10.90 ± 0.14
Fisher Scope,-7.38 ± 0.15,-9.13 ± 0.14,-10.93 ± 0.13
Temperature Scope,-8.25 ± 0.15,-10.31 ± 0.14,-11.48 ± 0.12


### Most Influential Token

In [9]:
# Filter entries with "loo_rank" (influence ranking vs LOO comparison)
loo_rank_data = {k: v for k, v in data.items() if "loo_rank" in k and "Llama-3.2-1B" in k}

# Map raw method names to display names (same as lambada table)
METHOD_DISPLAY = {
    "Random": "random",
    "Temperature": "Temperature Scope",
    "gradient_x_input": "Input x Grad",
    "Semantic": "Semantic Scope",
    "IG": "Integrated Grads",
    "Fisher": "Fisher Scope",
    
}

# Build lookup: label key → mean_ranking_pct ± SEM string
def fmt_loo_val(entry):
    mean_pct = entry.get("mean_ranking_pct")
    sem_pct = entry.get("sem_mean_ranking_pct")
    if mean_pct is not None and sem_pct is not None:
        return f"{mean_pct:.1f} ± {sem_pct:.1f}%"
    elif mean_pct is not None:
        return f"{mean_pct:.1f}%"
    return None

method_to_val = {}
for label, entry in loo_rank_data.items():
    parts = label.split("__")
    raw_method = parts[1].replace("_lambada_loo_rank", "") if len(parts) >= 2 else label
    display_method = METHOD_DISPLAY.get(raw_method, raw_method)
    method_to_val[display_method] = fmt_loo_val(entry)

# Build table with same row order as lambada: random, Temperature Scope, Semantic Scope, Gradient Input
# row_order = ["random", "Semantic Scope", "Gradient Input", "Temperature Scope"]
row_order = ["random","Integrated Grads", "Semantic Scope", "Input x Grad", "Temperature Scope", "Fisher Scope"]
loo_table = pd.DataFrame(
    {"average ranking": [method_to_val.get(m) for m in row_order]},
    index=row_order,
)
loo_table.index.name = "method"
loo_table

,average ranking
method,
random,None
Integrated Grads,None
Semantic Scope,None
Input x Grad,None
Temperature Scope,None
Fisher Scope,None


In [8]:
print(loo_table.to_markdown())

| method            | average ranking   |
|:------------------|:------------------|
| random            |                   |
| Integrated Grads  |                   |
| Semantic Scope    |                   |
| Input x Grad      |                   |
| Temperature Scope |                   |
| Fisher Scope      |                   |
